In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from imblearn.over_sampling import ADASYN
from sklearn.model_selection import train_test_split
from scipy.stats import pointbiserialr
import xgboost as xgb
from gplearn.genetic import SymbolicRegressor
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, precision_score, recall_score
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, Ridge
from sklearn.feature_selection import SelectFromModel
import xgboost as xgb


import warnings
warnings.filterwarnings("ignore")

In [2]:
Cards = pd.read_csv("./data/cards_data.csv")
Transactions = pd.read_csv("./data/transactions_data.csv")
Users = pd.read_csv("./data/users_data.csv")

In [3]:
Cards['credit_limit'] = Cards['credit_limit'].replace('[\$,]', '', regex=True).astype(float)
Users['yearly_income'] = Users['yearly_income'].replace('[\$,]', '', regex=True).astype(float)
Users['total_debt'] = Users['total_debt'].replace('[\$,]', '', regex=True).astype(float)
Users['per_capita_income'] = Users['per_capita_income'].replace('[\$,]', '', regex=True).astype(float)
Transactions['amount'] = Transactions['amount'].replace('[\$,]', '', regex=True).astype(float)

# 合併三表
Trans_merge = pd.DataFrame()
Trans_merge = Transactions.merge(Cards, left_on="card_id", right_on="id", suffixes=("_txn","_card"))
Trans_merge = Trans_merge.merge(Users, left_on="client_id_txn", right_on="id", suffixes=("","_user"))

### 這裡因為我們前面 EDA 發現每小時交量跟我們要檢測的 is_fraud 會有很大的關係，所以另外定義一個新變數加入預測變數。

In [4]:
Trans_merge['date'] = pd.to_datetime(Trans_merge['date'])

## 每小時交易量
Trans_merge['hour'] = Trans_merge['date'].dt.floor('H')

hourly_volume = (
    Trans_merge
    .groupby(['client_id_txn', 'hour'])['amount']
    .sum()
    .reset_index()
    .rename(columns={'amount': 'hourly_transaction_volume'})
)

if 'hourly_transaction_volume' in Trans_merge.columns:
    Trans_merge = Trans_merge.drop(columns=['hourly_transaction_volume'])
    
Trans_merge = Trans_merge.merge(
    hourly_volume,
    on=['client_id_txn', 'hour'],
    how='left'
)

In [5]:
with open('./data/train_fraud_labels.json', 'r') as f:
    fraud_data = json.load(f)

fraud_dict = fraud_data['target']

fraud_dict = {int(k): v for k, v in fraud_dict.items()}

Trans_merge['id_txn'] = Trans_merge['id_txn'].astype(int)

Trans_merge['is_fraud'] = Trans_merge['id_txn'].map(fraud_dict)
Trans_merge = Trans_merge[Trans_merge['is_fraud'].notna()].copy()
Trans_merge['is_fraud'] = Trans_merge['is_fraud'].map({'Yes': 1, 'No': 0})

In [6]:
Trans_merge.info()

print(Trans_merge['is_fraud'].value_counts())

<class 'pandas.core.frame.DataFrame'>
Index: 8914963 entries, 0 to 13305912
Data columns (total 42 columns):
 #   Column                     Dtype         
---  ------                     -----         
 0   id_txn                     int64         
 1   date                       datetime64[ns]
 2   client_id_txn              int64         
 3   card_id                    int64         
 4   amount                     float64       
 5   use_chip                   object        
 6   merchant_id                int64         
 7   merchant_city              object        
 8   merchant_state             object        
 9   zip                        float64       
 10  mcc                        int64         
 11  errors                     object        
 12  id_card                    int64         
 13  client_id_card             int64         
 14  card_brand                 object        
 15  card_type                  object        
 16  card_number                int64        

In [7]:
import time
from tqdm import tqdm

tqdm.pandas()

start_time = time.time()

df = Trans_merge.copy()
TARGET_COLUMN = 'is_fraud'

print("==== 移除遺失值過高欄位 ====")
missing_ratio = df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.7].index
print(f"以下欄位遺失超過 70%：{list(cols_to_drop)}")
df.drop(columns=cols_to_drop, inplace=True)
print(f"已移除 {len(cols_to_drop)} 個欄位。\n")

print("==== 將所有欄位轉為數值 ====")
for col in tqdm(df.columns, desc="Converting to numeric"):
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n=== 遺失值填補 ===")
cols_to_impute = df.columns[df.isnull().any()]
print(f"需要填補遺失值的欄位數：{len(cols_to_impute)}")

from sklearn.impute import SimpleImputer

for col in cols_to_impute:
    try:
        imputer = SimpleImputer(strategy="median")
        df[[col]] = imputer.fit_transform(df[[col]])
    except Exception as e:
        print(f"⚠️ 欄位 {col} 無法使用 median 填補，改用常數 0。錯誤：{e}")
        df[col] = df[col].fillna(0)

# 使用魯棒法和 min-max 標準化數據
print("==== \n數據標準化 ====")
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

print("RobustScaler 執行中...")
X_robust = pd.DataFrame(RobustScaler().fit_transform(X), columns=X.columns)

print("MinMaxScaler 執行中...")
X_scaled = pd.DataFrame(MinMaxScaler().fit_transform(X_robust), columns=X.columns)

print("==== \ntrain-test split ====")
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
print(f"訓練集: {len(X_train)}  | 測試集: {len(X_test)}\n")

print("==== ADASYN 平衡資料 ====")
t0 = time.time()
adasyn = ADASYN(sampling_strategy='minority', random_state=42, n_neighbors=5)
X_train_res, y_train_res = adasyn.fit_resample(X_train, y_train)
print(f"ADASYN 完成，耗時 {time.time() - t0:.1f} 秒")
print(f"ADASYN 後資料量: {len(X_train_res)}")
print(f"舞弊比例: {y_train_res.mean():.4f}")

print("\n總耗時：", time.time() - start_time, "秒")


==== 移除遺失值過高欄位 ====
以下欄位遺失超過 70%：['errors']
已移除 1 個欄位。

==== 將所有欄位轉為數值 ====


Converting to numeric: 100%|███████████████████████████████████████████████████████████| 41/41 [00:41<00:00,  1.00s/it]



=== 遺失值填補 ===
需要填補遺失值的欄位數：12
⚠️ 欄位 use_chip 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 merchant_city 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 merchant_state 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 card_brand 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 card_type 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 expires 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 has_chip 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 acct_open_date 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 card_on_dark_web 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 gender 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
⚠️ 欄位 address 無法使用 median 填補，改用常數 0。錯誤：Columns must be same length as key
==== 
數據標準化 ====
RobustScaler 執行中...
MinMaxScaler 執行中...
==== 
train-test split ====
訓練集: 7131970  | 測試集: 1782993

==== A

In [8]:
initial_features = X_train_res.columns.tolist()
feature_ranks = pd.DataFrame(index=initial_features)

# ============================================================
# 2. Lasso
# ============================================================
print("=== 1. Lasso 特徵選擇 ===")

lasso = Lasso(
    alpha=0.001,
    max_iter=30000,
    tol=1e-3,
    random_state=34
)
lasso.fit(X_train_res, y_train_res)

lasso_coeffs_sorted = (
    pd.Series(lasso.coef_, index=initial_features)
    .abs()
    .sort_values(ascending=False)
)

# 計算非零係數特徵
lasso_nonzero_features = lasso_coeffs_sorted[lasso_coeffs_sorted > 1e-6]

# 統一顯示格式
N_TOP_K = min(50, len(initial_features))
lasso_topk = lasso_coeffs_sorted.index[:N_TOP_K].tolist()

print(f"非零係數特徵數：{len(lasso_nonzero_features)}")
print(f"Lasso Top {N_TOP_K} 特徵：{lasso_topk[:10]}")


# ============================================================
# 2. Ridge
# ============================================================
print("\n=== 2. Ridge 特徵選擇 ===")

ridge = Ridge(
    alpha=1.0,
    max_iter=30000,
    tol=1e-3,
    random_state=34
)
ridge.fit(X_train_res, y_train_res)

ridge_coeffs_sorted = (
    pd.Series(ridge.coef_, index=initial_features)
    .abs()
    .sort_values(ascending=False)
)

# Ridge 不會產生零係數，所以我們只顯示前 K 名
ridge_topk = ridge_coeffs_sorted.index[:N_TOP_K].tolist()

print(f"非零係數特徵數：{len(ridge_coeffs_sorted)}")
print(f"Ridge Top {N_TOP_K} 特徵：{ridge_topk[:10]}")


# ============================================================
# 3. XGBOOST
# ============================================================
print("\n=== 3. XGBoost 特徵選擇 ===")

xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",
    enable_categorical=False
)
xgb_model.fit(X_train_res, y_train_res)

raw_importance = xgb_model.get_booster().get_score(importance_type="gain")

mapped_importance = pd.Series(0, index=initial_features)

for key, val in raw_importance.items():
    if key.startswith("f") and key[1:].isdigit():
        idx = int(key[1:])
        col_name = initial_features[idx]
    else:
        col_name = key  # 直接就是欄位名稱
        
    if col_name in mapped_importance.index:
        mapped_importance[col_name] = val

importance_sorted = mapped_importance.sort_values(ascending=False)

feature_ranks["XGBoost_Rank"] = importance_sorted.rank(ascending=False)

xgb_selected = importance_sorted[importance_sorted > 0].index.tolist()
print(f"XGBoost 重要性>0 的特徵數：{len(xgb_selected)}")
print(f"XGBoost Top 10: {importance_sorted.index[:10].tolist()}")


# ============================================================
# 4. 三模型聯集
# ============================================================
print("\n=== 4. 三種方法 Top-K 聯集 ===")

N_TOP_K = 50
combined_features = []
seen = set()

for feat_list in [lasso_coeffs_sorted.index[:N_TOP_K],
                  ridge_coeffs_sorted.index[:N_TOP_K],
                  importance_sorted.index[:N_TOP_K]]:
    for f in feat_list:
        if f not in seen:
            combined_features.append(f)
            seen.add(f)

print(f"三模型 Top {N_TOP_K} 聯集 → 共 {len(combined_features)} 個特徵")
print(f"聯集前 10: {combined_features[:10]}")

print("\n=== 5. 最終 XGBoost Top 5 精選 ===")

X_combined = X_train_res[combined_features]
xgb_model.fit(X_combined, y_train_res)

final_raw_importance = xgb_model.get_booster().get_score(importance_type="gain")

mapped_final = pd.Series(0, index=combined_features)

for key, val in final_raw_importance.items():
    if key.startswith("f") and key[1:].isdigit():
        idx = int(key[1:])
        col_name = combined_features[idx]
    else:
        col_name = key  # XGB 直接輸出特徵名稱
    
    if col_name in mapped_final.index:
        mapped_final[col_name] = val

sorted_final = mapped_final.sort_values(ascending=False)
final_features = sorted_final.index[:5].tolist()

print("\n🎯 最終選到的 Top 5 特徵：")
print(final_features)

X_train_final = X_train_res[final_features]
X_test_final  = X_test[final_features]

print("\n=== 特徵選擇完成 ===")


=== 1. Lasso 特徵選擇 ===
非零係數特徵數：13
Lasso Top 40 特徵：['hourly_transaction_volume', 'mcc', 'credit_limit', 'zip', 'num_credit_cards', 'merchant_id', 'longitude', 'id_txn', 'yearly_income', 'birth_month']

=== 2. Ridge 特徵選擇 ===
非零係數特徵數：40
Ridge Top 40 特徵：['hourly_transaction_volume', 'id_txn', 'birth_year', 'hour', 'current_age', 'date', 'mcc', 'credit_limit', 'amount', 'per_capita_income']

=== 3. XGBoost 特徵選擇 ===
XGBoost 重要性>0 的特徵數：26
XGBoost Top 10: ['hourly_transaction_volume', 'zip', 'mcc', 'hour', 'merchant_id', 'date', 'longitude', 'latitude', 'num_credit_cards', 'id_txn']

=== 4. 三種方法 Top-K 聯集 ===
三模型 Top 50 聯集 → 共 40 個特徵
聯集前 10: ['hourly_transaction_volume', 'mcc', 'credit_limit', 'zip', 'num_credit_cards', 'merchant_id', 'longitude', 'id_txn', 'yearly_income', 'birth_month']

=== 5. 最終 XGBoost Top 5 精選 ===

🎯 最終選到的 Top 5 特徵：
['hourly_transaction_volume', 'zip', 'mcc', 'hour', 'merchant_id']

=== 特徵選擇完成 ===
